# EARLY-DIAGNOSED ADHD RL MODEL

### original version

In [4]:
"""
EARLY-DIAGNOSED ADHD RL MODEL - VERSION 2.0 (Stable & TWAS-aligned)
"""
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.distributions import Categorical
import copy

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CONFIG = {
    'n_samples_per_episode': 64,
    'n_episodes_per_phase': 150,
    'n_phases': 5,
    'prune_sparsity': 0.94,           # realistic early-ADHD pruning
    'baseline_stress': 0.42,
    'stim_stress_reduction': 0.68,
    'stim_dopamine_boost': 1.45,
    'ketamine_regrow_fraction': 0.58,
    'gamma': 0.98,
}

def generate_blobs(n, noise=0.8):
    centers = np.array([[-3,-3],[3,3],[-3,3],[3,-3]])
    labels = np.random.randint(0,4,n)
    data = centers[labels] + np.random.randn(n,2)*noise
    return torch.tensor(data,dtype=torch.float32), torch.tensor(labels,dtype=torch.long)

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 4)
        self.stress = CONFIG['baseline_stress']
    def set_stress(self, s): self.stress = s
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = h + torch.randn_like(h) * self.stress
        h = torch.relu(self.fc2(h))
        h = h + torch.randn_like(h) * self.stress
        return self.fc3(h)
    def act(self, state):
        logits = self.forward(state)
        dist = Categorical(logits=logits.softmax(-1))
        a = dist.sample()
        return a.item(), dist.log_prob(a)

class Pruner:
    def __init__(self, model):
        self.model = model
        self.masks = {n: torch.ones_like(p) for n,p in model.named_parameters() if 'weight' in n}
    def prune(self):
        for n,p in self.model.named_parameters():
            if n in self.masks:
                thresh = torch.quantile(p.data.abs().flatten(), CONFIG['prune_sparsity'])
                self.masks[n] = (p.data.abs() >= thresh).float()
                p.data *= self.masks[n]
    def apply(self):
        with torch.no_grad():
            for n,p in self.model.named_parameters():
                if n in self.masks: p.data *= self.masks[n]
    def regrow(self, frac):
        for n,p in self.model.named_parameters():
            if n in self.masks:
                pruned = (self.masks[n]==0)
                if not pruned.any(): continue
                g = p.grad.abs() if p.grad is not None else torch.ones_like(p)
                scores = g[pruned]
                n_regrow = int(frac * pruned.sum().item())
                if n_regrow == 0: continue
                _,idx = torch.topk(scores, n_regrow)
                flat = torch.where(pruned.flatten())[0][idx]
                self.masks[n].flatten()[flat] = 1
                p.data.flatten()[flat] = torch.randn(n_regrow)*0.025
        self.apply()

class StimMod:
    def __init__(self, model):
        self.model = model
        self.on = False
    def toggle(self, on=True):
        self.on = on
        self.model.set_stress(CONFIG['baseline_stress']*(1-CONFIG['stim_stress_reduction']) if on else CONFIG['baseline_stress'])

def train_phase(model, pruner, stim, phase, ketamine=False):
    opt = optim.Adam(model.parameters(), lr=0.001*(1.8 if stim.on else 1))
    model.set_stress(CONFIG['baseline_stress']*(1-CONFIG['stim_stress_reduction']) if stim.on else CONFIG['baseline_stress'])
    history = []
    for ep in range(CONFIG['n_episodes_per_phase']):
        s, labs = generate_blobs(CONFIG['n_samples_per_episode'])
        logps, rs = [], []
        for i in range(len(s)):
            a, lp = model.act(s[i].unsqueeze(0))
            r = 1.0 if a == labs[i].item() else 0.0
            if stim.on: r *= CONFIG['stim_dopamine_boost']
            logps.append(lp)
            rs.append(r)
        returns = []
        R = 0
        for r in reversed(rs):
            R = r + CONFIG['gamma']*R
            returns.insert(0, R)
        returns = torch.tensor(returns)
        adv = (returns - returns.mean()) / (returns.std() + 1e-8)  # advantage baseline
        loss = sum(-lp * a for lp,a in zip(logps,adv)) / len(logps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        pruner.apply()
        if ketamine and ep % 10 == 0:
            pruner.regrow(CONFIG['ketamine_regrow_fraction'])
        history.append(np.mean([r>0 for r in rs])*100)  # accuracy %
    return np.mean(history[-40:])

# ====================== RUN ======================
base = PolicyNet()
pru = Pruner(base)
pru.prune()

# 1. Untreated
m1 = copy.deepcopy(base)
p1 = Pruner(m1); p1.masks = {k:v.clone() for k,v in pru.masks.items()}; p1.apply()
s1 = StimMod(m1); s1.toggle(False)
unt = [train_phase(m1,p1,s1,p) for p in range(5)]
print(f"Untreated adult accuracy: {unt[-1]:.1f}%")

# 2. Chronic stimulants
m2 = copy.deepcopy(base)
p2 = Pruner(m2); p2.masks = {k:v.clone() for k,v in pru.masks.items()}; p2.apply()
s2 = StimMod(m2); s2.toggle(True)
chrn = [train_phase(m2,p2,s2,p) for p in range(5)]
print(f"Chronic stimulants adult accuracy: {chrn[-1]:.1f}%")

# 3. Late intervention
m3 = copy.deepcopy(base)
p3 = Pruner(m3); p3.masks = {k:v.clone() for k,v in pru.masks.items()}; p3.apply()
s3 = StimMod(m3); s3.toggle(False)
for p in range(4): train_phase(m3,p3,s3,p)
s3.toggle(True)
late_stim = train_phase(m3,p3,s3,4)
print(f"Late stimulants only: {late_stim:.1f}%")
p3.regrow(CONFIG['ketamine_regrow_fraction'])
late_combo = train_phase(m3,p3,s3,4,ketamine=True)
print(f"Late stim + ketamine-like: {late_combo:.1f}%")

Untreated adult accuracy: 25.9%
Chronic stimulants adult accuracy: 30.1%
Late stimulants only: 25.3%
Late stim + ketamine-like: 35.2%


### improved

All original classes (`PolicyNet`, `Pruner`, `StimMod`), the `train_phase` function, `generate_blobs`, and `CONFIG` are completely unchanged. The additions are:

**`compute_sparsity(pruner)`** — returns the fraction of zero entries across all weight masks for a given pruner instance.

**20-seed loop** — each seed reinitializes `torch.manual_seed` and `np.random.seed`, then runs all four arms identically to the original single-run block. After each arm finishes, the final accuracy (last phase), final sparsity (from the pruner masks), and final stress (from `model.stress`) are recorded.

**Six output tables:**

- **Table 1** — per-seed final accuracy for all four arms.
- **Table 2** — per-seed final sparsity (fraction of pruned weights) for all four arms.
- **Table 3** — per-seed final stress value for all four arms.
- **Table 4** — summary accuracy: mean, std, min, max, and 95% CI bounds (two-tailed t-distribution, df=19, \(t_{\text{crit}} = 2.093\)).
- **Table 5** — summary sparsity with the same statistics and CI.
- **Table 6** — summary stress with the same statistics and CI.

In [5]:
"""
EARLY-DIAGNOSED ADHD RL MODEL - VERSION 2.0 (Stable & TWAS-aligned)
"""
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.distributions import Categorical
import copy

CONFIG = {
    'n_samples_per_episode': 64,
    'n_episodes_per_phase': 150,
    'n_phases': 5,
    'prune_sparsity': 0.94,           # realistic early-ADHD pruning
    'baseline_stress': 0.42,
    'stim_stress_reduction': 0.68,
    'stim_dopamine_boost': 1.45,
    'ketamine_regrow_fraction': 0.58,
    'gamma': 0.98,
}

def generate_blobs(n, noise=0.8):
    centers = np.array([[-3,-3],[3,3],[-3,3],[3,-3]])
    labels = np.random.randint(0,4,n)
    data = centers[labels] + np.random.randn(n,2)*noise
    return torch.tensor(data,dtype=torch.float32), torch.tensor(labels,dtype=torch.long)

class PolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 4)
        self.stress = CONFIG['baseline_stress']
    def set_stress(self, s): self.stress = s
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = h + torch.randn_like(h) * self.stress
        h = torch.relu(self.fc2(h))
        h = h + torch.randn_like(h) * self.stress
        return self.fc3(h)
    def act(self, state):
        logits = self.forward(state)
        dist = Categorical(logits=logits.softmax(-1))
        a = dist.sample()
        return a.item(), dist.log_prob(a)

class Pruner:
    def __init__(self, model):
        self.model = model
        self.masks = {n: torch.ones_like(p) for n,p in model.named_parameters() if 'weight' in n}
    def prune(self):
        for n,p in self.model.named_parameters():
            if n in self.masks:
                thresh = torch.quantile(p.data.abs().flatten(), CONFIG['prune_sparsity'])
                self.masks[n] = (p.data.abs() >= thresh).float()
                p.data *= self.masks[n]
    def apply(self):
        with torch.no_grad():
            for n,p in self.model.named_parameters():
                if n in self.masks: p.data *= self.masks[n]
    def regrow(self, frac):
        for n,p in self.model.named_parameters():
            if n in self.masks:
                pruned = (self.masks[n]==0)
                if not pruned.any(): continue
                g = p.grad.abs() if p.grad is not None else torch.ones_like(p)
                scores = g[pruned]
                n_regrow = int(frac * pruned.sum().item())
                if n_regrow == 0: continue
                _,idx = torch.topk(scores, n_regrow)
                flat = torch.where(pruned.flatten())[0][idx]
                self.masks[n].flatten()[flat] = 1
                p.data.flatten()[flat] = torch.randn(n_regrow)*0.025
        self.apply()

class StimMod:
    def __init__(self, model):
        self.model = model
        self.on = False
    def toggle(self, on=True):
        self.on = on
        self.model.set_stress(CONFIG['baseline_stress']*(1-CONFIG['stim_stress_reduction']) if on else CONFIG['baseline_stress'])

def train_phase(model, pruner, stim, phase, ketamine=False):
    opt = optim.Adam(model.parameters(), lr=0.001*(1.8 if stim.on else 1))
    model.set_stress(CONFIG['baseline_stress']*(1-CONFIG['stim_stress_reduction']) if stim.on else CONFIG['baseline_stress'])
    history = []
    for ep in range(CONFIG['n_episodes_per_phase']):
        s, labs = generate_blobs(CONFIG['n_samples_per_episode'])
        logps, rs = [], []
        for i in range(len(s)):
            a, lp = model.act(s[i].unsqueeze(0))
            r = 1.0 if a == labs[i].item() else 0.0
            if stim.on: r *= CONFIG['stim_dopamine_boost']
            logps.append(lp)
            rs.append(r)
        returns = []
        R = 0
        for r in reversed(rs):
            R = r + CONFIG['gamma']*R
            returns.insert(0, R)
        returns = torch.tensor(returns)
        adv = (returns - returns.mean()) / (returns.std() + 1e-8)  # advantage baseline
        loss = sum(-lp * a for lp,a in zip(logps,adv)) / len(logps)
        opt.zero_grad()
        loss.backward()
        opt.step()
        pruner.apply()
        if ketamine and ep % 10 == 0:
            pruner.regrow(CONFIG['ketamine_regrow_fraction'])
        history.append(np.mean([r>0 for r in rs])*100)  # accuracy %
    return np.mean(history[-40:])


# ====================== HELPER FUNCTIONS ======================

def compute_sparsity(pruner):
    """Fraction of weights that are zeroed out (masked)."""
    total = 0
    zeros = 0
    for n, mask in pruner.masks.items():
        total += mask.numel()
        zeros += (mask == 0).sum().item()
    return zeros / total if total > 0 else 0.0


# ====================== RUN 20 SEEDS ======================

N_SEEDS = 20
ARM_NAMES = ['Untreated', 'Chronic Stimulants', 'Late Stim Only', 'Late Stim + Ketamine']

all_results = {arm: {'accuracy': [], 'sparsity': [], 'stress': []} for arm in ARM_NAMES}

for seed in range(N_SEEDS):
    torch.manual_seed(seed)
    np.random.seed(seed)

    base = PolicyNet()
    pru = Pruner(base)
    pru.prune()

    # 1. Untreated
    m1 = copy.deepcopy(base)
    p1 = Pruner(m1); p1.masks = {k: v.clone() for k, v in pru.masks.items()}; p1.apply()
    s1 = StimMod(m1); s1.toggle(False)
    unt = [train_phase(m1, p1, s1, p) for p in range(5)]
    all_results['Untreated']['accuracy'].append(unt[-1])
    all_results['Untreated']['sparsity'].append(compute_sparsity(p1))
    all_results['Untreated']['stress'].append(m1.stress)

    # 2. Chronic stimulants
    m2 = copy.deepcopy(base)
    p2 = Pruner(m2); p2.masks = {k: v.clone() for k, v in pru.masks.items()}; p2.apply()
    s2 = StimMod(m2); s2.toggle(True)
    chrn = [train_phase(m2, p2, s2, p) for p in range(5)]
    all_results['Chronic Stimulants']['accuracy'].append(chrn[-1])
    all_results['Chronic Stimulants']['sparsity'].append(compute_sparsity(p2))
    all_results['Chronic Stimulants']['stress'].append(m2.stress)

    # 3. Late intervention
    m3 = copy.deepcopy(base)
    p3 = Pruner(m3); p3.masks = {k: v.clone() for k, v in pru.masks.items()}; p3.apply()
    s3 = StimMod(m3); s3.toggle(False)
    for p in range(4):
        train_phase(m3, p3, s3, p)
    s3.toggle(True)
    late_stim = train_phase(m3, p3, s3, 4)
    all_results['Late Stim Only']['accuracy'].append(late_stim)
    all_results['Late Stim Only']['sparsity'].append(compute_sparsity(p3))
    all_results['Late Stim Only']['stress'].append(m3.stress)

    p3.regrow(CONFIG['ketamine_regrow_fraction'])
    late_combo = train_phase(m3, p3, s3, 4, ketamine=True)
    all_results['Late Stim + Ketamine']['accuracy'].append(late_combo)
    all_results['Late Stim + Ketamine']['sparsity'].append(compute_sparsity(p3))
    all_results['Late Stim + Ketamine']['stress'].append(m3.stress)

    print(f"[Seed {seed:2d}] Untreated={unt[-1]:.2f}%  Chronic Stim={chrn[-1]:.2f}%  "
          f"Late Stim={late_stim:.2f}%  Late+Ket={late_combo:.2f}%  |  "
          f"Sparsity: U={compute_sparsity(p1):.4f} C={compute_sparsity(p2):.4f} "
          f"LS={all_results['Late Stim Only']['sparsity'][-1]:.4f} "
          f"LK={all_results['Late Stim + Ketamine']['sparsity'][-1]:.4f}  |  "
          f"Stress: U={m1.stress:.4f} C={m2.stress:.4f} LS={m3.stress:.4f} LK={m3.stress:.4f}")


# ====================== DETAILED TABLES ======================

T_CRIT_19 = 2.093  # t critical value for df=19, two-tailed alpha=0.05

def ci_bounds(vals):
    a = np.array(vals)
    m = a.mean()
    s = a.std(ddof=1)
    se = s / np.sqrt(len(a))
    return m, s, a.min(), a.max(), m - T_CRIT_19 * se, m + T_CRIT_19 * se


print("\n")
print("=" * 110)
print("  DETAILED RESULTS ACROSS 20 SEEDS")
print("=" * 110)

# ---- TABLE 1: Per-Seed Accuracy ----
print("\n" + "-" * 110)
print("  TABLE 1: Per-Seed Final Accuracy (%)")
print("-" * 110)
hdr = f"{'Seed':>4} | {'Untreated':>14} | {'Chronic Stim':>14} | {'Late Stim Only':>16} | {'Late Stim+Ket':>16}"
print(hdr)
print("-" * len(hdr))
for i in range(N_SEEDS):
    print(f"{i:4d} | {all_results['Untreated']['accuracy'][i]:13.2f}% "
          f"| {all_results['Chronic Stimulants']['accuracy'][i]:13.2f}% "
          f"| {all_results['Late Stim Only']['accuracy'][i]:15.2f}% "
          f"| {all_results['Late Stim + Ketamine']['accuracy'][i]:15.2f}%")
print("-" * len(hdr))

# ---- TABLE 2: Per-Seed Final Sparsity ----
print("\n" + "-" * 110)
print("  TABLE 2: Per-Seed Final Sparsity (fraction of pruned weights)")
print("-" * 110)
hdr2 = f"{'Seed':>4} | {'Untreated':>14} | {'Chronic Stim':>14} | {'Late Stim Only':>16} | {'Late Stim+Ket':>16}"
print(hdr2)
print("-" * len(hdr2))
for i in range(N_SEEDS):
    print(f"{i:4d} | {all_results['Untreated']['sparsity'][i]:14.6f} "
          f"| {all_results['Chronic Stimulants']['sparsity'][i]:14.6f} "
          f"| {all_results['Late Stim Only']['sparsity'][i]:16.6f} "
          f"| {all_results['Late Stim + Ketamine']['sparsity'][i]:16.6f}")
print("-" * len(hdr2))

# ---- TABLE 3: Per-Seed Final Stress ----
print("\n" + "-" * 110)
print("  TABLE 3: Per-Seed Final Stress (noise magnitude)")
print("-" * 110)
hdr3 = f"{'Seed':>4} | {'Untreated':>14} | {'Chronic Stim':>14} | {'Late Stim Only':>16} | {'Late Stim+Ket':>16}"
print(hdr3)
print("-" * len(hdr3))
for i in range(N_SEEDS):
    print(f"{i:4d} | {all_results['Untreated']['stress'][i]:14.6f} "
          f"| {all_results['Chronic Stimulants']['stress'][i]:14.6f} "
          f"| {all_results['Late Stim Only']['stress'][i]:16.6f} "
          f"| {all_results['Late Stim + Ketamine']['stress'][i]:16.6f}")
print("-" * len(hdr3))

# ---- TABLE 4: Summary – Accuracy ----
print("\n" + "-" * 110)
print("  TABLE 4: Summary Statistics – Final Accuracy (%)  [95% CI, t-distribution, df=19]")
print("-" * 110)
sh = (f"{'Arm':<22} | {'Mean':>8} | {'Std':>8} | {'Min':>8} | {'Max':>8} "
      f"| {'CI Lower':>10} | {'CI Upper':>10}")
print(sh)
print("-" * len(sh))
for arm in ARM_NAMES:
    m, s, mn, mx, lo, hi = ci_bounds(all_results[arm]['accuracy'])
    print(f"{arm:<22} | {m:8.2f} | {s:8.2f} | {mn:8.2f} | {mx:8.2f} | {lo:10.2f} | {hi:10.2f}")
print("-" * len(sh))

# ---- TABLE 5: Summary – Sparsity ----
print("\n" + "-" * 110)
print("  TABLE 5: Summary Statistics – Final Sparsity  [95% CI, t-distribution, df=19]")
print("-" * 110)
sh2 = (f"{'Arm':<22} | {'Mean':>10} | {'Std':>10} | {'Min':>10} | {'Max':>10} "
       f"| {'CI Lower':>10} | {'CI Upper':>10}")
print(sh2)
print("-" * len(sh2))
for arm in ARM_NAMES:
    m, s, mn, mx, lo, hi = ci_bounds(all_results[arm]['sparsity'])
    print(f"{arm:<22} | {m:10.6f} | {s:10.6f} | {mn:10.6f} | {mx:10.6f} | {lo:10.6f} | {hi:10.6f}")
print("-" * len(sh2))

# ---- TABLE 6: Summary – Stress ----
print("\n" + "-" * 110)
print("  TABLE 6: Summary Statistics – Final Stress  [95% CI, t-distribution, df=19]")
print("-" * 110)
sh3 = (f"{'Arm':<22} | {'Mean':>10} | {'Std':>10} | {'Min':>10} | {'Max':>10} "
       f"| {'CI Lower':>10} | {'CI Upper':>10}")
print(sh3)
print("-" * len(sh3))
for arm in ARM_NAMES:
    m, s, mn, mx, lo, hi = ci_bounds(all_results[arm]['stress'])
    print(f"{arm:<22} | {m:10.6f} | {s:10.6f} | {mn:10.6f} | {mx:10.6f} | {lo:10.6f} | {hi:10.6f}")
print("-" * len(sh3))

print("\n" + "=" * 110)
print(f"  N_SEEDS = {N_SEEDS}  |  CI method: t-distribution (df={N_SEEDS-1}, "
      f"t_crit={T_CRIT_19})  |  Confidence level: 95%")
print("=" * 110)

[Seed  0] Untreated=26.09%  Chronic Stim=28.91%  Late Stim=25.86%  Late+Ket=36.84%  |  Sparsity: U=0.9400 C=0.9400 LS=0.9400 LK=0.0001  |  Stress: U=0.4200 C=0.1344 LS=0.1344 LK=0.1344
[Seed  1] Untreated=25.90%  Chronic Stim=30.86%  Late Stim=26.05%  Late+Ket=47.34%  |  Sparsity: U=0.9400 C=0.9400 LS=0.9400 LK=0.0001  |  Stress: U=0.4200 C=0.1344 LS=0.1344 LK=0.1344
[Seed  2] Untreated=25.70%  Chronic Stim=29.49%  Late Stim=26.84%  Late+Ket=42.30%  |  Sparsity: U=0.9400 C=0.9400 LS=0.9400 LK=0.0001  |  Stress: U=0.4200 C=0.1344 LS=0.1344 LK=0.1344
[Seed  3] Untreated=24.02%  Chronic Stim=28.67%  Late Stim=26.99%  Late+Ket=40.08%  |  Sparsity: U=0.9400 C=0.9400 LS=0.9400 LK=0.0001  |  Stress: U=0.4200 C=0.1344 LS=0.1344 LK=0.1344
[Seed  4] Untreated=25.51%  Chronic Stim=28.75%  Late Stim=26.64%  Late+Ket=33.52%  |  Sparsity: U=0.9400 C=0.9400 LS=0.9400 LK=0.0001  |  Stress: U=0.4200 C=0.1344 LS=0.1344 LK=0.1344
[Seed  5] Untreated=25.00%  Chronic Stim=27.93%  Late Stim=28.98%  Late+Ket

# The End